# 08 - Visualization

The `orthograph.visualization` package renders orthograph data structures for human consumption. It supports two output formats:

| Format | Function suffix | Dependencies | Use case |
|--------|----------------|-------------|----------|
| **Mermaid** | `_to_mermaid` | None | Embeddable in markdown, notebooks, docs |
| **Plain text** | `_to_text` | None | Terminal output, logs, CI artifacts |

Three input types can be visualized:

| Input | Mermaid | Text |
|-------|---------|------|
| `GraphDataModel` (schema) | `model_to_mermaid` / `display_mermaid` | `model_to_text` |
| `GraphProfile` (inspection) | -- | `profile_to_text` |
| `ValidationResult` (validation) | -- | `result_to_text` |

This notebook demonstrates every renderer using the filmography domain.

In [ ]:
from typing import Optional

import networkx as nx

from orthograph import (
    Cardinality,
    GraphDataModel,
    NodeModel,
    RelationshipModel,
)
from orthograph.extensions.networkx import NetworkxInspector
from orthograph.extensions.validation import validate_profile
from orthograph.visualization import render
from orthograph.visualization.mermaid import display_mermaid, model_to_mermaid
from orthograph.visualization.text import model_to_text, profile_to_text, result_to_text

## Setup: model, graph, profile, and validation result

We define the filmography model, build a NetworkX graph with intentionally incomplete data, inspect it, and validate it. This gives us all three input types for the renderers.

In [ ]:
class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"
    name: str
    age: int
    email: Optional[str] = None


class Movie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    year: int
    rating: Optional[float] = None


class City(NodeModel):
    __label__ = "City"
    __uid_field__ = "name"
    name: str
    country: str


class ActedIn(RelationshipModel):
    __label__ = "ACTED_IN"
    __source_type__ = Person
    __target_type__ = Movie
    role: str


class Directed(RelationshipModel):
    __label__ = "DIRECTED"
    __source_type__ = Person
    __target_type__ = Movie


class LivesIn(RelationshipModel):
    __label__ = "LIVES_IN"
    __source_type__ = Person
    __target_type__ = City
    __source_cardinality__ = Cardinality.ONE
    __target_cardinality__ = Cardinality.ZERO_OR_MORE


model = GraphDataModel(
    name="Filmography",
    node_types=[Person, Movie, City],
    relationship_types=[ActedIn, Directed, LivesIn],
)

In [ ]:
# Build a graph with intentionally incomplete data
G = nx.MultiDiGraph()

G.add_node("p1", __label__="Person", name="Alice", age=30, email="alice@example.com")
G.add_node("p2", __label__="Person", name="Bob", age=45)
G.add_node("p3", __label__="Person", name="Charlie")  # missing 'age'

G.add_node("m1", __label__="Movie", title="The Matrix", year=1999, rating=8.7)
G.add_node("m2", __label__="Movie", title="Inception", year=2010)

G.add_node("c1", __label__="City", name="Los Angeles", country="USA")

G.add_edge("p1", "m1", __label__="ACTED_IN", role="Trinity")
G.add_edge("p2", "m1", __label__="ACTED_IN", role="Morpheus")
G.add_edge("p1", "m2", __label__="ACTED_IN", role="Ariadne")
G.add_edge("p2", "m2", __label__="DIRECTED")
G.add_edge("p1", "c1", __label__="LIVES_IN")
G.add_edge("p2", "c1", __label__="LIVES_IN")

# Inspect and validate
profile = NetworkxInspector(G).inspect()
result = validate_profile(profile, model)

print(f"Model:    {model.name} ({len(model.node_types)} node types, {len(model.relationship_types)} relationship types)")
print(f"Profile:  {sum(ntp.count for ntp in profile.node_type_profiles.values())} nodes, {sum(rtp.count for rtp in profile.rel_type_profiles.values())} relationships")
print(f"Result:   {'PASS' if result.is_valid else 'FAIL'} ({len(result.errors)} errors, {len(result.warnings)} warnings)")

## Model visualization: Mermaid

`model_to_mermaid` renders the schema definition as a Mermaid diagram. Node boxes show properties with type, required/optional markers, and UID highlighting. Edges show the relationship type, any properties, and source/target cardinality.

In [ ]:
print(model_to_mermaid(model))

## Model visualization: plain text

`model_to_text` renders the same information as a structured text table. Useful for terminal output and logs where Mermaid rendering is not available.

In [ ]:
print(model_to_text(model))

## Profile visualization: plain text

`profile_to_text` renders a `GraphProfile` as a text table showing instance counts, property completeness percentages, mandatory/partial flags, observed types, and cardinality statistics.

In [ ]:
print(profile_to_text(profile))

## Validation result visualization: plain text

`result_to_text` renders a `ValidationResult` as a severity-coded summary. Issues are grouped by entity, with `[ERROR]`, `[WARNING]`, and `[INFO]` prefixes.

In [ ]:
print(result_to_text(result))

## The `render()` dispatcher

The `render()` function provides a single entry point. It dispatches based on the input type and the requested format (`"mermaid"` or `"text"`).

In [ ]:
# Model as mermaid
output = render(model, format="mermaid")
print(f"render(model, format='mermaid') -> {len(output)} chars")
print()

# Profile as text
output = render(profile, format="text")
print(f"render(profile, format='text') -> {len(output)} chars")
print()

# Result as text
output = render(result, format="text")
print(f"render(result, format='text') -> {len(output)} chars")

## Inline Mermaid rendering with `display_mermaid`

`display_mermaid` renders a Mermaid diagram as an inline image in a Jupyter notebook. It uses the [mermaid.ink](https://mermaid.ink) service to convert the diagram text to a PNG image (requires an internet connection).

It accepts a raw Mermaid string, a `GraphDataModel`, or a `GraphProfile` -- the conversion to Mermaid text is handled automatically.

In [ ]:
# Render the model schema as an inline diagram
display_mermaid(model)

In [ ]:
# Also works with raw Mermaid strings
display_mermaid("graph LR\n    A --> B --> C")

## Side-by-side: schema vs observed

A useful pattern is to render the model schema and the observed profile side by side, so you can compare what the schema *expects* with what the data *contains*.

In [ ]:
print("=" * 60)
print("SCHEMA (model definition)")
print("=" * 60)
print(model_to_text(model))

print("=" * 60)
print("OBSERVED (graph profile)")
print("=" * 60)
print(profile_to_text(profile))

print("=" * 60)
print("VALIDATION")
print("=" * 60)
print(result_to_text(result))